In [1]:
import torch
import torch.nn as nn
from torchvision.models import resnet18

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 100)
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model = model.to(device)
model.eval()


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [2]:
from AircraftDataset import AircraftDataset
from torchvision import transforms
from torch.utils.data import DataLoader

transform_test = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = AircraftDataset('data/fgvc-aircraft/train.csv', 'data/fgvc-aircraft/images', transform_test)

test_dataset = AircraftDataset('data/fgvc-aircraft/test.csv', 'data/fgvc-aircraft/images', transform_test)
test_loader = DataLoader(test_dataset, batch_size=32)


In [3]:
from tqdm import tqdm


correct, total = 0, 0
with torch.no_grad():
    for images, labels in tqdm(test_loader):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test accuracy: {correct / total:.4f}")


In [4]:
from OneImage import predict_image


class_mapping = train_dataset.class_to_idx

result1 = predict_image("data/fgvc-aircraft/images/0063281.jpg", model, train_dataset.class_to_idx, device)
result2 = predict_image("data/3.jpg", model, train_dataset.class_to_idx, device)

print("Predicted class:", result2)


Predicted class: Spitfire


In [6]:
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk
import torch
from torchvision import transforms
import os
import random

# Модель, трансформ, класс-маппинг
model.eval()
class_to_idx = train_dataset.class_to_idx  # или test_dataset.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# UI
root = tk.Tk()
root.title("Aircraft Classifier")

image_label = tk.Label(root)
image_label.pack()

result_label = tk.Label(root, text="", font=("Arial", 14))
result_label.pack()

similar_frame = tk.Frame(root)
similar_frame.pack()

def load_image():
    file_path = filedialog.askopenfilename()
    if not file_path:
        return

    # Оригинал
    img = Image.open(file_path).convert('RGB')
    img_resized = img.resize((256, 256))
    tk_img = ImageTk.PhotoImage(img_resized)
    image_label.configure(image=tk_img)
    image_label.image = tk_img

    # Предсказание
    input_tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(input_tensor)
        predicted_idx = output.argmax(1).item()
        predicted_class = idx_to_class[predicted_idx]

    result_label.config(text=f"Распознанная модель самолёта: {predicted_class}")

    # Похожие изображения
    for widget in similar_frame.winfo_children():
        widget.destroy()

    class_df = train_dataset.data[train_dataset.data['Classes'] == predicted_class]
    sample_paths = class_df.sample(min(5, len(class_df)))['filename'].tolist()

    for path in sample_paths:
        full_path = os.path.join(train_dataset.img_dir, path)
        sim_img = Image.open(full_path).convert('RGB').resize((128, 128))
        tk_sim = ImageTk.PhotoImage(sim_img)
        lbl = tk.Label(similar_frame, image=tk_sim)
        lbl.image = tk_sim
        lbl.pack(side="left", padx=5)

tk.Button(root, text="Загрузить изображение", command=load_image).pack(pady=10)

root.mainloop()
